In [1]:
import pandas as pd
import numpy as np
from scipy.stats import randint
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.metrics import (accuracy_score, classification_report,
                             confusion_matrix, roc_auc_score)
from sklearn.model_selection import RandomizedSearchCV, GridSearchCV
from sklearn.preprocessing import StandardScaler

# Seed-Based Prediction Dataframe

This notebook creates a prediction dataframe where stat differences are calculated as higher seed stats minus lower seed stats, with labels indicating if the higher seed wins (1) or loses (0).

## Load and Explore the Data

In [2]:
# Load advanced team season stats
df_advanced = pd.read_csv('data/teams_advanced.csv')

# Clean team names
df_advanced['team'] = (
    df_advanced['team']
    .astype(str)
    .str.replace(' ', ' ', regex=False)
    .str.replace('*', '', regex=False)
    .str.strip()
)

df_advanced['team'] = df_advanced['team'].replace({
    'Albany (NY)': 'Albany'
})

df_advanced = df_advanced[df_advanced['team'] != 'School'].copy()

# Convert numeric columns
num_cols = df_advanced.columns.drop('team')
df_advanced[num_cols] = df_advanced[num_cols].apply(pd.to_numeric, errors='coerce')

df_advanced = df_advanced.loc[:, ~df_advanced.columns.str.contains('^Unnamed')]

print("Advanced Team Stats Shape:", df_advanced.shape)
print("\nSample advanced stats:")
print(df_advanced.head())

Advanced Team Stats Shape: (680, 29)

Sample advanced stats:
       team   G  wins  losses  win_pct    SRS   SOS  wins.1  losses.1  wins.2  \
0    Albany  33    24       9    0.727  -0.34 -5.18      15         1      12   
1   Arizona  38    34       4    0.895  24.33  7.41      16         2      17   
2  Arkansas  36    27       9    0.750  14.07  6.79      13         5      17   
3    Baylor  34    24      10    0.706  17.85  9.33      11         7      16   
4   Belmont  33    22      11    0.667   0.35 -2.77      11         5      12   

   ...    TS%  TRB%  AST%  STL%  BLK%   eFG%  TOV%  ORB%  FT/FGA  year  
0  ...  0.546  53.5  47.9   9.2   5.5  0.499  16.2  32.6   0.296  2015  
1  ...  0.575  56.8  52.3  10.8   9.8  0.535  14.2  34.5   0.338  2015  
2  ...  0.539  50.2  59.2  10.9  12.0  0.500  14.3  35.7   0.270  2015  
3  ...  0.530  55.9  61.1  12.2  11.2  0.497  16.4  42.4   0.260  2015  
4  ...  0.589  50.3  58.7   9.4   5.5  0.567  17.8  29.0   0.225  2015  

[5 rows x 29 

In [3]:
# Load tournament games
games = pd.read_csv('data/tournament_games.csv')

games['winner'] = games['winner'].replace({'Albany (NY)': 'Albany'})
games['loser'] = games['loser'].replace({'Albany (NY)': 'Albany'})

print("Tournament Games Shape:", games.shape)
print("\nSample games:")
print(games.head())

Tournament Games Shape: (629, 5)

Sample games:
   season         winner  winner_seed       loser  loser_seed
0    2015      Villanova            1   Lafayette          16
1    2015       NC State            8         LSU           9
2    2015  Northern Iowa            5     Wyoming          12
3    2015     Louisville            4   UC Irvine          13
4    2015         Dayton           11  Providence           6


## Identify Seed Information and Determine Higher/Lower Seeds

In [4]:
# Merge team stats with games
# For each game, identify winner and loser stats
games_w = games.merge(
    df_advanced,
    left_on=['winner', 'season'],
    right_on=['team', 'year'],
    how='left'
)

games_wl = games_w.merge(
    df_advanced,
    left_on=['loser', 'season'],
    right_on=['team', 'year'],
    how='left',
    suffixes=('_winner', '_loser')
)

# Identify which team is higher seed (lower seed number)
games_wl['higher_seed_team'] = games_wl.apply(
    lambda row: 'winner' if row['winner_seed'] < row['loser_seed'] else 'loser',
    axis=1
)

games_wl['higher_seed_value'] = games_wl[['winner_seed', 'loser_seed']].min(axis=1)
games_wl['lower_seed_value'] = games_wl[['winner_seed', 'loser_seed']].max(axis=1)

# Boolean: did higher seed win?
games_wl['higher_seed_won'] = (games_wl['higher_seed_team'] == 'winner').astype(int)

print("Sample games with seed info:")
print(games_wl[['winner', 'winner_seed', 'loser', 'loser_seed', 'higher_seed_team', 'higher_seed_won']].head(10))

Sample games with seed info:
           winner  winner_seed          loser  loser_seed higher_seed_team  \
0       Villanova            1      Lafayette          16           winner   
1        NC State            8            LSU           9           winner   
2   Northern Iowa            5        Wyoming          12           winner   
3      Louisville            4      UC Irvine          13           winner   
4          Dayton           11     Providence           6            loser   
5        Oklahoma            3         Albany          14           winner   
6  Michigan State            7        Georgia          10           winner   
7        Virginia            2        Belmont          15           winner   
8        NC State            8      Villanova           1            loser   
9      Louisville            4  Northern Iowa           5           winner   

   higher_seed_won  
0                1  
1                1  
2                1  
3                1  
4      

## Calculate Stat Differences by Seed

In [5]:
# Define the statistical features to calculate differences for
features = [
    'SRS', 'win_pct', 'eFG%', 'TS%', 'TRB%', 'AST%', 'TOV%', 'ORtg'
]

# Calculate differences: higher seed stats - lower seed stats
for col in features:
    games_wl[f'{col}_diff'] = games_wl.apply(
        lambda row: 
            (row[f'{col}_winner'] - row[f'{col}_loser']) 
            if row['higher_seed_team'] == 'winner' 
            else (row[f'{col}_loser'] - row[f'{col}_winner']),
        axis=1
    )

# Seed difference: higher seed value - lower seed value (always negative, typically -15 to 0)
games_wl['seed_diff'] = games_wl['higher_seed_value'] - games_wl['lower_seed_value']

print("Sample calculated differences:")
print(games_wl[[f'{col}_diff' for col in features] + ['seed_diff', 'higher_seed_won']].head(10))

Sample calculated differences:
   SRS_diff  win_pct_diff  eFG%_diff  TS%_diff  TRB%_diff  AST%_diff  \
0     26.50         0.311     -0.004     0.000        3.2        3.3   
1       NaN           NaN        NaN       NaN        NaN        NaN   
2      8.73         0.172      0.033     0.033        2.0      -13.9   
3     14.49         0.132     -0.043    -0.032        0.6       -8.5   
4      2.73        -0.103     -0.046    -0.039        4.3       -3.3   
5     18.99        -0.041     -0.006    -0.014       -2.4        0.7   
6      5.04         0.056      0.047     0.020        0.9        8.7   
7     21.38         0.215     -0.061    -0.046        5.7       -5.2   
8      9.02         0.306      0.060     0.063       -0.8       15.8   
9      5.58        -0.136     -0.082    -0.081       -0.1       -5.5   

   TOV%_diff  ORtg_diff  seed_diff  higher_seed_won  
0       -1.1        4.0        -15                1  
1        NaN        NaN         -1                1  
2       -1.0  

## Combine Features and Labels

In [6]:
# Build the final seed-based dataframe
feature_cols = sorted([c for c in games_wl.columns if c.endswith('_diff')])

# Create label: 1 if higher seed won, 0 if lower seed won
games_wl['win_label'] = games_wl['higher_seed_won']

# Select final columns
seed_df = games_wl[feature_cols + ['win_label', 'season']].copy()

print(f"Seed-based dataframe shape: {seed_df.shape}")
print(f"\nFeature columns: {feature_cols}")
print(f"\nFirst few rows:")
print(seed_df.head(10))

Seed-based dataframe shape: (629, 11)

Feature columns: ['AST%_diff', 'ORtg_diff', 'SRS_diff', 'TOV%_diff', 'TRB%_diff', 'TS%_diff', 'eFG%_diff', 'seed_diff', 'win_pct_diff']

First few rows:
   AST%_diff  ORtg_diff  SRS_diff  TOV%_diff  TRB%_diff  TS%_diff  eFG%_diff  \
0        3.3        4.0     26.50       -1.1        3.2     0.000     -0.004   
1        NaN        NaN       NaN        NaN        NaN       NaN        NaN   
2      -13.9        8.8      8.73       -1.0        2.0     0.033      0.033   
3       -8.5       -0.7     14.49       -1.4        0.6    -0.032     -0.043   
4       -3.3        0.3      2.73       -0.8        4.3    -0.039     -0.046   
5        0.7       -1.2     18.99       -1.1       -2.4    -0.014     -0.006   
6        8.7        6.9      5.04       -1.6        0.9     0.020      0.047   
7       -5.2        0.9     21.38       -4.4        5.7    -0.046     -0.061   
8       15.8        8.9      9.02        0.5       -0.8     0.063      0.060   
9       

## Validate and Display Results

In [7]:
# Check for missing values
missing_values = seed_df.isnull().sum()
print("Missing values:")
print(missing_values[missing_values > 0] if missing_values.sum() > 0 else "No missing values!")

# Label distribution
print(f"\nLabel distribution:")
print(seed_df['win_label'].value_counts())
print(f"\nHigher seed win rate: {seed_df['win_label'].mean():.1%}")

# Data types
print(f"\nData types:")
print(seed_df.dtypes)

Missing values:
AST%_diff       95
ORtg_diff       95
SRS_diff        95
TOV%_diff       95
TRB%_diff       95
TS%_diff        95
eFG%_diff       95
win_pct_diff    95
dtype: int64

Label distribution:
win_label
1    444
0    185
Name: count, dtype: int64

Higher seed win rate: 70.6%

Data types:
AST%_diff       float64
ORtg_diff       float64
SRS_diff        float64
TOV%_diff       float64
TRB%_diff       float64
TS%_diff        float64
eFG%_diff       float64
seed_diff         int64
win_pct_diff    float64
win_label         int64
season            int64
dtype: object


In [8]:
# Remove rows with missing values and save to CSV
seed_df_clean = seed_df.dropna()

print(f"\nRows removed: {len(seed_df) - len(seed_df_clean)}")
print(f"Final dataframe shape: {seed_df_clean.shape}")

seed_df_clean.to_csv('data/tournament_seed_ml.csv', index=False)
print("\nDataframe saved to 'data/tournament_seed_ml.csv'")

# Display final summary
print(f"\nFinal dataframe summary:")
print(seed_df_clean.describe())


Rows removed: 95
Final dataframe shape: (534, 11)

Dataframe saved to 'data/tournament_seed_ml.csv'

Final dataframe summary:
        AST%_diff   ORtg_diff    SRS_diff   TOV%_diff   TRB%_diff    TS%_diff  \
count  534.000000  534.000000  534.000000  534.000000  534.000000  534.000000   
mean     0.944007    3.144757    9.050955   -0.578464    0.983146    0.008498   
std      7.082520    6.481768    8.414595    1.990410    3.219531    0.035228   
min    -17.500000  -13.400000   -8.260000   -6.800000   -9.500000   -0.089000   
25%     -3.775000   -1.175000    2.782500   -1.900000   -1.100000   -0.016000   
50%      1.200000    2.600000    7.410000   -0.500000    0.800000    0.007000   
75%      5.300000    7.375000   13.430000    0.800000    3.000000    0.031750   
max     23.700000   23.300000   38.630000    5.600000   10.400000    0.110000   

        eFG%_diff   seed_diff  win_pct_diff   win_label       season  
count  534.000000  534.000000    534.000000  534.000000   534.000000  
m

## Random Forest Model on Seed-Based Dataframe

In [9]:
# Load the cleaned seed-based dataframe
df_seed = pd.read_csv('data/tournament_seed_ml.csv')

print(f"Seed-based dataframe shape: {df_seed.shape}")
print(f"Columns: {df_seed.columns.tolist()}")

Seed-based dataframe shape: (534, 11)
Columns: ['AST%_diff', 'ORtg_diff', 'SRS_diff', 'TOV%_diff', 'TRB%_diff', 'TS%_diff', 'eFG%_diff', 'seed_diff', 'win_pct_diff', 'win_label', 'season']


In [10]:
# Define features and split train/test by season
features = sorted([c for c in df_seed.columns if c.endswith('_diff')])

# Train/test split by season
train = df_seed[df_seed['season'] <= 2022]
test  = df_seed[df_seed['season'] >= 2023]

X_train, y_train = train[features], train['win_label']
X_test,  y_test  = test[features],  test['win_label']

print(f"Train rows: {len(train)} | Test rows: {len(test)}")
print(f"\nFeatures: {features}")

Train rows: 377 | Test rows: 157

Features: ['AST%_diff', 'ORtg_diff', 'SRS_diff', 'TOV%_diff', 'TRB%_diff', 'TS%_diff', 'eFG%_diff', 'seed_diff', 'win_pct_diff']


In [11]:
# Set up hyperparameter grid for RandomizedSearchCV
param_dist = {
    'n_estimators':      (100, 800),
    'max_depth': [None] + list(range(5, 31)),
    'min_samples_split': randint(2, 20),
    'min_samples_leaf': randint(1, 8),
    'max_features':      ['sqrt', 'log2', None],
    'bootstrap':         [True, False]
}

rf = RandomForestClassifier(random_state=42, n_jobs=-1)

# Randomized search — tries 50 random combinations, 5-fold CV
search = RandomizedSearchCV(
    estimator=rf,
    param_distributions=param_dist,
    n_iter=50,           # number of random combos to try
    cv=5,                # 5-fold cross-validation
    scoring='accuracy',   # optimize for accuracy
    random_state=42,
    n_jobs=-1,
    verbose=1
)

print("Starting RandomizedSearchCV...")

Starting RandomizedSearchCV...


In [12]:
search.fit(X_train, y_train)

print(f"\nBest Parameters: {search.best_params_}")
print(f"Best CV Accuracy:     {search.best_score_:.4f}")

# Evaluate best model on test set
best_rf = search.best_estimator_

y_pred  = best_rf.predict(X_test)
y_proba = best_rf.predict_proba(X_test)[:, 1]

print(f"\nTest Accuracy:  {accuracy_score(y_test, y_pred):.4f}")
print(f"Test ROC-AUC:   {roc_auc_score(y_test, y_proba):.4f}")

cm = confusion_matrix(y_test, y_pred)
# For sklearn confusion_matrix with labels [0, 1], the layout is:
# [[TN, FP],
#  [FN, TP]]
print(f"\nConfusion Matrix:\n{cm}")

tn, fp, fn, tp = cm.ravel()
win1_pct_correct = tp / (tp + fn) if (tp + fn) else 0.0
win0_pct_correct = tn / (tn + fp) if (tn + fp) else 0.0
print(f"\nPercent of win_label=1 correctly predicted: {win1_pct_correct:.1%}")
print(f"Percent of win_label=0 correctly predicted: {win0_pct_correct:.1%}")
print(f"\nClassification Report:\n{classification_report(y_test, y_pred)}")

Fitting 5 folds for each of 50 candidates, totalling 250 fits

Best Parameters: {'bootstrap': True, 'max_depth': 10, 'max_features': 'sqrt', 'min_samples_leaf': 1, 'min_samples_split': 8, 'n_estimators': 800}
Best CV Accuracy:     0.7480

Test Accuracy:  0.7325
Test ROC-AUC:   0.7360

Confusion Matrix:
[[16 30]
 [12 99]]

Percent of win_label=1 correctly predicted: 89.2%
Percent of win_label=0 correctly predicted: 34.8%

Classification Report:
              precision    recall  f1-score   support

           0       0.57      0.35      0.43        46
           1       0.77      0.89      0.82       111

    accuracy                           0.73       157
   macro avg       0.67      0.62      0.63       157
weighted avg       0.71      0.73      0.71       157



In [13]:
# Feature importances from best model
importances = sorted(zip(features, best_rf.feature_importances_),
                     key=lambda x: x[1], reverse=True)
print("\nFeature Importances:")
for feat, imp in importances:
    print(f"  {feat:<15} {imp:.4f}")


Feature Importances:
  SRS_diff        0.2523
  win_pct_diff    0.1296
  TRB%_diff       0.1124
  AST%_diff       0.1079
  ORtg_diff       0.0924
  TS%_diff        0.0851
  eFG%_diff       0.0789
  seed_diff       0.0736
  TOV%_diff       0.0677


## Support Vector Machine (SVM) Model with GridSearchCV

In [14]:
# Scale features for SVM (important for SVM performance)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Features scaled for SVM")
print(f"X_train_scaled shape: {X_train_scaled.shape}")
print(f"X_test_scaled shape: {X_test_scaled.shape}")

Features scaled for SVM
X_train_scaled shape: (377, 9)
X_test_scaled shape: (157, 9)


In [15]:
# Set up hyperparameter grid for SVM GridSearchCV
param_grid = {
    'C': [0.1, 1, 10, 100],
    'kernel': ['linear', 'rbf', 'poly'],
    'gamma': ['scale', 'auto'],
    'degree': [2, 3]
}

svm = SVC(random_state=42, probability=True)

# GridSearchCV for SVM
svm_search = GridSearchCV(
    estimator=svm,
    param_grid=param_grid,
    cv=5,                # 5-fold cross-validation
    scoring='accuracy',  # optimize for accuracy
    n_jobs=-1,
    verbose=1
)

print("Starting GridSearchCV for SVM...")

Starting GridSearchCV for SVM...


In [16]:
svm_search.fit(X_train_scaled, y_train)

print(f"\nBest Parameters: {svm_search.best_params_}")
print(f"Best CV Accuracy:     {svm_search.best_score_:.4f}")

# Evaluate best model on test set
best_svm = svm_search.best_estimator_

y_pred_svm  = best_svm.predict(X_test_scaled)
y_proba_svm = best_svm.predict_proba(X_test_scaled)[:, 1]

print(f"\nTest Accuracy:  {accuracy_score(y_test, y_pred_svm):.4f}")
print(f"Test ROC-AUC:   {roc_auc_score(y_test, y_proba_svm):.4f}")
print(f"\nConfusion Matrix:\n{confusion_matrix(y_test, y_pred_svm)}")
print(f"\nClassification Report:\n{classification_report(y_test, y_pred_svm)}")

Fitting 5 folds for each of 48 candidates, totalling 240 fits

Best Parameters: {'C': 1, 'degree': 2, 'gamma': 'scale', 'kernel': 'linear'}
Best CV Accuracy:     0.7640

Test Accuracy:  0.7134
Test ROC-AUC:   0.7051

Confusion Matrix:
[[14 32]
 [13 98]]

Classification Report:
              precision    recall  f1-score   support

           0       0.52      0.30      0.38        46
           1       0.75      0.88      0.81       111

    accuracy                           0.71       157
   macro avg       0.64      0.59      0.60       157
weighted avg       0.68      0.71      0.69       157

